# 📊 均线策略回测系统

**双均线策略 + 趋势过滤器 + ATR过滤器**

---

## 📋 功能说明

本 Notebook 提供完整的均线策略回测功能：

1. ✅ **数据加载** - 从 `data/adjusted/` 加载5只不同行业的复权股价数据
2. ✅ **技术指标计算** - 支持 MA/EMA、ATR 计算
3. ✅ **信号生成** - 双均线策略 + 趋势过滤器 + ATR过滤器
4. ✅ **模拟回测** - 执行策略回测，支持手续费和滑点
5. ✅ **量化指标** - 收益类、风险类、综合类、交易质量类指标
6. ✅ **可视化** - K线图、均线、买卖信号、净值曲线
7. ✅ **BUY-HOLD对比** - 与买入持有策略进行比较

---

## 🚀 快速开始

1. **运行 Cell 2** - 安装依赖库
2. **运行 Cell 3** - 配置策略参数
3. **运行 Cell 4** - 加载股票数据
4. **运行 Cell 5** - 计算技术指标和生成信号
5. **运行 Cell 6** - 执行回测
6. **运行 Cell 7** - 查看可视化图表
7. **运行 Cell 8** - 查看量化指标

---


## 🔧 Cell 1: 安装依赖库

In [ ]:
# 安装依赖库（首次运行需要执行）
import sys
import subprocess

def install_package(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

# 安装所需库
packages = ['pandas', 'numpy', 'plotly', 'matplotlib', 'ipywidgets']

for pkg in packages:
    try:
        __import__(pkg)
        print(f'✅ {pkg} 已安装')
    except ImportError:
        print(f'📦 正在安装 {pkg}...')
        install_package(pkg)
        print(f'✅ {pkg} 安装完成')

print('\n🎉 所有依赖库已就绪！')

## ⚙️ Cell 2: 策略参数配置

**仿照 HTML 界面左侧边栏设计**

---

### 📈 股票选择

In [ ]:
# ============================================================================
# 策略参数配置（用户可自定义）
# ============================================================================

# 股票列表（5只不同行业的股票）
STOCKS = {
    '300750.SZ': {'name': '宁德时代', 'industry': '科技/新能源'},
    '601318.SH': {'name': '中国平安', 'industry': '金融'},
    '600519.SH': {'name': '贵州茅台', 'industry': '消费'},
    '601857.SH': {'name': '中国石油', 'industry': '能源'},
    '002594.SZ': {'name': '比亚迪', 'industry': '制造'}
}

# 选择要回测的股票（修改这里的代码即可切换股票）
SELECTED_STOCK = '600519.SH'  # 可选: '300750.SZ', '601318.SH', '600519.SH', '601857.SH', '002594.SZ'

print('=' * 60)
print('📈 股票选择')
print('=' * 60)
print(f'当前选择: {STOCKS[SELECTED_STOCK]["name"]} ({SELECTED_STOCK})')
print(f'所属行业: {STOCKS[SELECTED_STOCK]["industry"]}')
print('=' * 60)

In [ ]:
# ============================================================================
# 回测参数配置
# ============================================================================

STRATEGY_PARAMS = {
    # 均线参数
    'short_window': 5,           # 短均线周期
    'long_window': 15,           # 长均线周期
    'ma_type': 'MA',             # 'MA' 或 'EMA'
    
    # 趋势过滤器
    'trend_filter': True,        # 是否启用趋势过滤器
    'trend_window': 120,         # 趋势过滤器周期
    
    # ATR过滤器
    'atr_filter': True,          # 是否启用ATR过滤器
    'atr_window': 14,           # ATR计算周期
    'atr_percentile': 20,       # ATR历史百分位阈值（P20）
    'atr_lookback': 100,        # ATR历史lookback天数
    
    # 回测参数
    'initial_capital': 100000,   # 初始资金（元）
    'commission': 0.001,        # 手续费率（0.1% = 0.001）
    'slippage': 0.001,          # 滑点（0.1% = 0.001）
    
    # 仓位管理
    'position_sizing': 'full',   # 'full' | 'fixed_shares' | 'fixed_ratio'
    'fixed_shares': 100,        # 固定数量
    'fixed_ratio': 0.2,         # 固定比例
    
    # 日期范围
    'start_date': '2024-07-03',
    'end_date': '2026-07-03',
}

print('=' * 60)
print('⚙️ 策略参数配置')
print('=' * 60)
print(f'\n📐 均线参数:')
print(f'  短均线周期: {STRATEGY_PARAMS["short_window"]}')
print(f'  长均线周期: {STRATEGY_PARAMS["long_window"]}')
print(f'  均线类型: {STRATEGY_PARAMS["ma_type"]}')

print(f'\n🔍 信号过滤器:')
print(f'  趋势过滤器: {"启用" if STRATEGY_PARAMS["trend_filter"] else "禁用"}')
print(f'  ATR过滤器: {"启用" if STRATEGY_PARAMS["atr_filter"] else "禁用"}')

print(f'\n💰 回测参数:')
print(f'  初始资金: {STRATEGY_PARAMS["initial_capital"]:,} 元')
print(f'  手续费率: {STRATEGY_PARAMS["commission"]*100:.2f}%')
print(f'  滑点: {STRATEGY_PARAMS["slippage"]*100:.2f}%')

print(f'\n📅 回测时间段:')
print(f'  开始日期: {STRATEGY_PARAMS["start_date"]}')
print(f'  结束日期: {STRATEGY_PARAMS["end_date"]}')
print('=' * 60)

## 📥 Cell 3: 加载股票数据

In [ ]:
# ============================================================================
# 加载股票数据
# ============================================================================

import os
import pandas as pd

def load_stock_data(ts_code, start_date=None, end_date=None):
    """
    加载股票数据
    
    参数：
        ts_code: 股票代码
        start_date: 起始日期（可选）
        end_date: 结束日期（可选）
    
    返回：
        处理后的DataFrame
    """
    # 文件映射
    file_map = {
        '300750.SZ': 'ningde_times_300750_daily_adjusted.csv',
        '601318.SH': 'ping_an_601318_daily_adjusted.csv',
        '600519.SH': 'moutai_600519_daily_adjusted.csv',
        '601857.SH': 'petro_china_601857_daily_adjusted.csv',
        '002594.SZ': 'byd_002594_daily_adjusted.csv'
    }
    
    filename = file_map.get(ts_code)
    filepath = f'data/adjusted/{filename}'
    
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'文件不存在: {filepath}')
    
    # 加载数据
    df = pd.read_csv(filepath)
    
    # 数据预处理
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    df = df.sort_values('trade_date').reset_index(drop=True)
    
    # 筛选日期范围
    if start_date:
        start_date = pd.to_datetime(start_date)
        df = df[df['trade_date'] >= start_date].copy()
    if end_date:
        end_date = pd.to_datetime(end_date)
        df = df[df['trade_date'] <= end_date].copy()
    
    df = df.reset_index(drop=True)
    
    return df

# 加载数据
print('[1/3] 加载数据...')
df = load_stock_data(
    SELECTED_STOCK,
    start_date=STRATEGY_PARAMS['start_date'],
    end_date=STRATEGY_PARAMS['end_date']
)

print(f'✅ 数据加载完成')
print(f'  股票: {STOCKS[SELECTED_STOCK]["name"]} ({SELECTED_STOCK})')
print(f'  日期范围: {df["trade_date"].min().date()} 至 {df["trade_date"].max().date()}')
print(f'  数据条数: {len(df)}')
print(f'\n前5行数据:')
display(df.head())

## 📊 Cell 4: 计算技术指标和生成信号

In [ ]:
# ============================================================================
# 计算技术指标
# ============================================================================

import numpy as np

def calculate_ma(prices, window):
    """计算移动平均"""
    return prices.rolling(window=window, min_periods=1).mean()

def calculate_ema(prices, window):
    """计算指数移动平均"""
    return prices.ewm(span=window, min_periods=1, adjust=False).mean()

def calculate_atr(df, window=14):
    """计算ATR（平均真实波幅）"""
    high = df['high']
    low = df['low']
    close = df['close']
    
    tr1 = high - low
    tr2 = abs(high - close.shift(1))
    tr3 = abs(low - close.shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(window=window, min_periods=1).mean()
    
    return atr

# 计算均线
print('[2/3] 计算技术指标...')

if STRATEGY_PARAMS['ma_type'] == 'MA':
    df['ma_short'] = calculate_ma(df['close'], STRATEGY_PARAMS['short_window'])
    df['ma_long'] = calculate_ma(df['close'], STRATEGY_PARAMS['long_window'])
else:
    df['ma_short'] = calculate_ema(df['close'], STRATEGY_PARAMS['short_window'])
    df['ma_long'] = calculate_ema(df['close'], STRATEGY_PARAMS['long_window'])

# 计算ATR
df['atr'] = calculate_atr(df, STRATEGY_PARAMS['atr_window'])

# 计算趋势过滤器均线
if STRATEGY_PARAMS['trend_filter']:
    df['ma_trend'] = calculate_ma(df['close'], STRATEGY_PARAMS['trend_window'])

print(f'✅ 技术指标计算完成')
print(f'  短均线: MA{STRATEGY_PARAMS["short_window"]}')
print(f'  长均线: MA{STRATEGY_PARAMS["long_window"]}')
print(f'  ATR: ATR{STRATEGY_PARAMS["atr_window"]}')

In [ ]:
# ============================================================================
# 生成交易信号
# ============================================================================

def generate_signals(df, params):
    """
    生成交易信号
    
    信号规则：
    - 买入（signal=1）：短均线上穿长均线（金叉）
    - 卖出（signal=-1）：短均线下穿长均线（死叉）
    
    过滤器：
    - 趋势过滤器：价格和短期均线都在长期趋势均线上方
    - ATR过滤器：ATR值处于历史高位时才交易
    """
    df = df.copy()
    df['signal'] = 0
    
    # 金叉和死叉识别
    df['golden_cross'] = (df['ma_short'] > df['ma_long']) & (df['ma_short'].shift(1) <= df['ma_long'].shift(1))
    df['death_cross'] = (df['ma_short'] < df['ma_long']) & (df['ma_short'].shift(1) >= df['ma_long'].shift(1))
    
    # 趋势过滤器
    if params['trend_filter']:
        trend_condition = (df['close'] > df['ma_trend']) & (df['ma_short'] > df['ma_trend'])
    else:
        trend_condition = True
    
    # ATR过滤器
    if params['atr_filter']:
        atr_threshold = df['atr'].rolling(window=params['atr_lookback']).quantile(params['atr_percentile']/100)
        atr_condition = df['atr'] > atr_threshold
    else:
        atr_condition = True
    
    # 生成信号
    for i in range(1, len(df)):
        if df.iloc[i]['golden_cross'] and trend_condition.iloc[i] and atr_condition.iloc[i]:
            df.iloc[i, df.columns.get_loc('signal')] = 1
        elif df.iloc[i]['death_cross']:
            df.iloc[i, df.columns.get_loc('signal')] = -1
    
    return df

# 生成信号
print('[3/3] 生成交易信号...')
df = generate_signals(df, STRATEGY_PARAMS)

# 统计信号数量
buy_signals = (df['signal'] == 1).sum()
sell_signals = (df['signal'] == -1).sum()

print(f'✅ 交易信号生成完成')
print(f'  买入信号: {buy_signals} 个')
print(f'  卖出信号: {sell_signals} 个')
print(f'\n信号分布:')
print(df['signal'].value_counts().sort_index())

## 💹 Cell 5: 执行回测

In [ ]:
# ============================================================================
# 执行策略回测
# ============================================================================

def run_backtest(df, params):
    """
    执行回测
    
    返回：
    - nav: 每日净值序列
    - returns: 每日收益率
    - trades: 交易记录
    """
    initial_capital = params['initial_capital']
    commission = params['commission']
    slippage = params['slippage']
    position_sizing = params.get('position_sizing', 'full')
    fixed_shares = params.get('fixed_shares', 100)
    fixed_ratio = params.get('fixed_ratio', 0.2)
    
    cash = initial_capital
    position = 0
    portfolio_value = []
    trades = []
    total_commission = 0
    total_slippage = 0
    
    for i in range(len(df)):
        signal = df.iloc[i]['signal']
        price = df.iloc[i]['close']
        date = df.iloc[i]['trade_date']
        
        if signal == 1 and position == 0:  # 买入
            # 计算滑点
            execution_price = price * (1 + slippage)
            
            # 计算买入数量
            if position_sizing == 'full':
                shares_to_buy = int(cash * (1 - commission) / execution_price)
            elif position_sizing == 'fixed_shares':
                shares_to_buy = min(fixed_shares, int(cash / execution_price))
            elif position_sizing == 'fixed_ratio':
                amount_to_invest = cash * fixed_ratio
                shares_to_buy = int(amount_to_invest * (1 - commission) / execution_price)
            else:
                shares_to_buy = 0
            
            if shares_to_buy > 0:
                cost = shares_to_buy * execution_price * (1 + commission)
                commission_cost = shares_to_buy * execution_price * commission
                slippage_cost = shares_to_buy * (execution_price - price)
                
                cash -= cost
                position += shares_to_buy
                total_commission += commission_cost
                total_slippage += slippage_cost
                
                trades.append({
                    'date': date,
                    'type': 'BUY',
                    'signal_price': price,
                    'execution_price': execution_price,
                    'shares': shares_to_buy,
                    'commission': commission_cost,
                    'slippage': slippage_cost,
                    'total_cost': commission_cost + slippage_cost,
                    'cash_after': cash,
                    'position_after': position
                })
        
        elif signal == -1 and position > 0:  # 卖出
            execution_price = price * (1 - slippage)
            
            if position_sizing == 'full':
                shares_to_sell = position
            elif position_sizing == 'fixed_shares':
                shares_to_sell = min(fixed_shares, position)
            elif position_sizing == 'fixed_ratio':
                shares_to_sell = int(position * fixed_ratio)
            else:
                shares_to_sell = 0
            
            if shares_to_sell > 0:
                revenue = shares_to_sell * execution_price * (1 - commission)
                commission_cost = shares_to_sell * execution_price * commission
                slippage_cost = shares_to_sell * (price - execution_price)
                
                cash += revenue
                position -= shares_to_sell
                total_commission += commission_cost
                total_slippage += slippage_cost
                
                trades.append({
                    'date': date,
                    'type': 'SELL',
                    'signal_price': price,
                    'execution_price': execution_price,
                    'shares': shares_to_sell,
                    'commission': commission_cost,
                    'slippage': slippage_cost,
                    'total_cost': commission_cost + slippage_cost,
                    'cash_after': cash,
                    'position_after': position
                })
        
        current_value = cash + position * price
        portfolio_value.append(current_value)
    
    nav = pd.Series(portfolio_value, index=df.index, name='nav')
    returns = nav.pct_change().fillna(0)
    
    return {
        'nav': nav,
        'returns': returns,
        'trades': pd.DataFrame(trades),
        'final_value': portfolio_value[-1] if portfolio_value else initial_capital,
        'transaction_costs': {
            'total_commission': total_commission,
            'total_slippage': total_slippage,
            'total_transaction_cost': total_commission + total_slippage
        }
    }

def run_buy_hold_backtest(df, params):
    """执行BUY-HOLD回测"""
    initial_capital = params['initial_capital']
    commission = params['commission']
    slippage = params['slippage']
    
    first_price = df.iloc[0]['close']
    execution_price = first_price * (1 + slippage)
    shares = int(initial_capital * (1 - commission) / execution_price)
    cash = initial_capital - shares * execution_price * (1 + commission)
    
    portfolio_value = []
    for i in range(len(df)):
        price = df.iloc[i]['close']
        current_value = cash + shares * price
        portfolio_value.append(current_value)
    
    nav = pd.Series(portfolio_value, index=df.index, name='buy_hold_nav')
    returns = nav.pct_change().fillna(0)
    
    return {
        'nav': nav,
        'returns': returns,
        'final_value': portfolio_value[-1] if portfolio_value else initial_capital,
        'shares': shares
    }

# 执行策略回测
print('=' * 60)
print('💹 执行回测')
print('=' * 60)

print('\n[1/2] 执行策略回测...')
backtest_result = run_backtest(df, STRATEGY_PARAMS)
print(f'✅ 策略回测完成')
print(f'  最终净值: {backtest_result["final_value"]:,.2f} 元')
print(f'  交易次数: {len(backtest_result["trades"])}')
print(f'\n  交易成本统计:')
print(f'    累计手续费: {backtest_result["transaction_costs"]["total_commission"]:,.2f} 元')
print(f'    累计滑点成本: {backtest_result["transaction_costs"]["total_slippage"]:,.2f} 元')
print(f'    累计交易成本: {backtest_result["transaction_costs"]["total_transaction_cost"]:,.2f} 元')

# 执行BUY-HOLD回测
print('\n[2/2] 执行BUY-HOLD回测...')
bh_result = run_buy_hold_backtest(df, STRATEGY_PARAMS)
print(f'✅ BUY-HOLD回测完成')
print(f'  最终净值: {bh_result["final_value"]:,.2f} 元')
print(f'  持有份额: {bh_result["shares"]} 股')

print('=' * 60)

## 📈 Cell 6: 可视化图表

**K线图 + 买卖信号 + 净值曲线**

In [ ]:
# ============================================================================
# 可视化图表
# ============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 创建子图：上方K线图，下方净值曲线
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.7, 0.3],
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=('📈 K线图 & 买卖信号', '💹 策略净值曲线')
)

# 1. K线图（用折线图模拟）
fig.add_trace(
    go.Scatter(
        x=df['trade_date'],
        y=df['close'],
        mode='lines',
        name='收盘价',
        line=dict(color='#f0f6fc', width=1.5)
    ),
    row=1, col=1
)

# 短均线
fig.add_trace(
    go.Scatter(
        x=df['trade_date'],
        y=df['ma_short'],
        mode='lines',
        name=f'MA{STRATEGY_PARAMS["short_window"]}',
        line=dict(color='#388bfd', width=1.5)
    ),
    row=1, col=1
)

# 长均线
fig.add_trace(
    go.Scatter(
        x=df['trade_date'],
        y=df['ma_long'],
        mode='lines',
        name=f'MA{STRATEGY_PARAMS["long_window"]}',
        line=dict(color='#f5a623', width=1.5)
    ),
    row=1, col=1
)

# 趋势过滤器均线
if STRATEGY_PARAMS['trend_filter']:
    fig.add_trace(
        go.Scatter(
            x=df['trade_date'],
            y=df['ma_trend'],
            mode='lines',
            name=f'MA{STRATEGY_PARAMS["trend_window"]} (趋势)',
            line=dict(color='#a8e6cf', width=1.5, dash='dash')
        ),
        row=1, col=1
    )

# 买入信号
buy_dates = df[df['signal'] == 1]['trade_date']
buy_prices = df[df['signal'] == 1]['close']

fig.add_trace(
    go.Scatter(
        x=buy_dates,
        y=buy_prices * 0.98,
        mode='markers',
        name='📈 买入信号',
        marker=dict(color='#ef5350', size=10, symbol='triangle-up')
    ),
    row=1, col=1
)

# 卖出信号
sell_dates = df[df['signal'] == -1]['trade_date']
sell_prices = df[df['signal'] == -1]['close']

fig.add_trace(
    go.Scatter(
        x=sell_dates,
        y=sell_prices * 1.02,
        mode='markers',
        name='📉 卖出信号',
        marker=dict(color='#26a69a', size=10, symbol='triangle-down')
    ),
    row=1, col=1
)

# 2. 净值曲线
fig.add_trace(
    go.Scatter(
        x=df['trade_date'],
        y=backtest_result['nav'],
        mode='lines',
        name='策略净值',
        line=dict(color='#388bfd', width=2)
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=df['trade_date'],
        y=bh_result['nav'],
        mode='lines',
        name='BUY-HOLD',
        line=dict(color='#f5a623', width=2, dash='dash')
    ),
    row=2, col=1
)

# 更新布局
fig.update_layout(
    title=f'{STOCKS[SELECTED_STOCK]["name"]} ({SELECTED_STOCK}) - 均线策略回测',
    paper_bgcolor='#161b22',
    plot_bgcolor='#0d1117',
    font=dict(color='#c9d1d9'),
    height=800,
    showlegend=True,
    legend=dict(bgcolor='rgba(22,27,34,0.8)')
)

fig.update_xaxes(gridcolor='#21262d', zerolinecolor='#21262d')
fig.update_yaxes(gridcolor='#21262d', zerolinecolor='#21262d')

fig.show()

## 📊 Cell 7: 量化指标计算

In [ ]:
# ============================================================================
# 计算量化指标
# ============================================================================

import numpy as np

def calculate_max_drawdown(nav):
    """计算最大回撤"""
    peak = nav.cummax()
    drawdown = (nav - peak) / peak
    max_dd = drawdown.min()
    return abs(max_dd)

def calculate_sharpe_ratio(returns, risk_free_rate=0.02):
    """计算夏普比率"""
    excess_returns = returns - risk_free_rate / 252
    if excess_returns.std() == 0:
        return np.nan
    sharpe = np.sqrt(252) * excess_returns.mean() / excess_returns.std()
    return sharpe

def calculate_win_rate(trades):
    """计算胜率"""
    if len(trades) == 0:
        return np.nan
    
    buy_prices = []
    win_count = 0
    total_sell_count = 0
    
    for idx, trade in trades.iterrows():
        if trade['type'] == 'BUY':
            buy_prices.append(trade['execution_price'])
        elif trade['type'] == 'SELL' and buy_prices:
            buy_price = buy_prices.pop(0)
            if trade['execution_price'] > buy_price:
                win_count += 1
            total_sell_count += 1
    
    if total_sell_count == 0:
        return np.nan
    
    return win_count / total_sell_count

def calculate_profit_loss_ratio(trades):
    """计算盈亏比"""
    if len(trades) == 0:
        return np.nan
    
    profits = []
    losses = []
    buy_prices = []
    
    for idx, trade in trades.iterrows():
        if trade['type'] == 'BUY':
            buy_prices.append(trade['execution_price'])
        elif trade['type'] == 'SELL' and buy_prices:
            buy_price = buy_prices.pop(0)
            profit = trade['execution_price'] - buy_price
            if profit > 0:
                profits.append(profit)
            elif profit < 0:
                losses.append(abs(profit))
    
    if not profits or not losses:
        return np.nan
    
    return np.mean(profits) / np.mean(losses)

# 计算指标
print('=' * 60)
print('📊 量化指标计算')
print('=' * 60)

# 策略指标
strategy_nav = backtest_result['nav']
strategy_returns = backtest_result['returns']
strategy_final = backtest_result['final_value']
strategy_trades = backtest_result['trades']

# BUY-HOLD指标
bh_nav = bh_result['nav']
bh_final = bh_result['final_value']

initial = strategy_nav.iloc[0]
trading_days = len(df)
years = trading_days / 252

# 收益类指标
strategy_total_return = (strategy_final - initial) / initial
bh_total_return = (bh_final - initial) / initial
excess_return = strategy_total_return - bh_total_return

strategy_annual_return = (1 + strategy_total_return) ** (1 / years) - 1 if years > 0 else np.nan
bh_annual_return = (1 + bh_total_return) ** (1 / years) - 1 if years > 0 else np.nan

# 风险类指标
strategy_max_dd = calculate_max_drawdown(strategy_nav)
bh_max_dd = calculate_max_drawdown(bh_nav)

# 综合类指标
strategy_sharpe = calculate_sharpe_ratio(strategy_returns)
bh_sharpe = calculate_sharpe_ratio(bh_nav.pct_change().fillna(0))

# 交易质量类指标
win_rate = calculate_win_rate(strategy_trades)
pl_ratio = calculate_profit_loss_ratio(strategy_trades)

# 打印报告
print('\n' + '=' * 60)
print('量化策略回测报告')
print('=' * 60)

print('\n------------------------------------------------------------')
print('收益类指标')
print('------------------------------------------------------------')
print(f'策略总收益率: {strategy_total_return*100:.2f}%')
print(f'BUY-HOLD总收益率: {bh_total_return*100:.2f}%')
print(f'超额收益: {excess_return*100:.2f}%')
print(f'策略年化收益率: {strategy_annual_return*100:.2f}%')
print(f'BUY-HOLD年化收益率: {bh_annual_return*100:.2f}%')

print('\n------------------------------------------------------------')
print('风险类指标')
print('------------------------------------------------------------')
print(f'策略最大回撤: {strategy_max_dd*100:.2f}%')
print(f'BUY-HOLD最大回撤: {bh_max_dd*100:.2f}%')

print('\n------------------------------------------------------------')
print('综合类指标')
print('------------------------------------------------------------')
print(f'策略夏普比率: {strategy_sharpe:.2f}')
print(f'BUY-HOLD夏普比率: {bh_sharpe:.2f}')

print('\n------------------------------------------------------------')
print('交易质量类指标')
print('------------------------------------------------------------')
print(f'交易次数: {len(strategy_trades)}')
if not np.isnan(win_rate):
    print(f'胜率: {win_rate*100:.2f}%')
if not np.isnan(pl_ratio):
    print(f'盈亏比: {pl_ratio:.2f}')

print('\n------------------------------------------------------------')
print('交易成本统计')
print('------------------------------------------------------------')
print(f'累计手续费: {backtest_result["transaction_costs"]["total_commission"]:,.2f} 元')
print(f'累计滑点成本: {backtest_result["transaction_costs"]["total_slippage"]:,.2f} 元')
print(f'累计交易成本: {backtest_result["transaction_costs"]["total_transaction_cost"]:,.2f} 元')
if len(strategy_trades) > 0:
    print(f'平均单次交易成本: {backtest_result["transaction_costs"]["total_transaction_cost"]/len(strategy_trades):,.2f} 元')

print('=' * 60)

## 📋 Cell 8: 交易记录明细

In [ ]:
# ============================================================================
# 交易记录明细
# ============================================================================

print('=' * 60)
print('📋 交易记录明细')
print('=' * 60)

if len(backtest_result['trades']) > 0:
    # 显示交易记录表格
    display(backtest_result['trades'])
    
    # 保存交易记录
    output_file = f'outputs/ma_backtest/reports/{STOCKS[SELECTED_STOCK]["name"]}_{SELECTED_STOCK}_trades.csv'
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    backtest_result['trades'].to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f'\n✅ 交易记录已保存: {output_file}')
else:
    print('\n⚠️ 暂无交易记录')

print('=' * 60)

## 🔄 Cell 9: 批量回测（5只股票）

**一键回测所有股票并生成对比报告**

In [ ]:
# ============================================================================
# 批量回测（5只股票）
# ============================================================================

print('=' * 60)
print('🔄 批量回测（5只股票）')
print('=' * 60)

results = []

for ts_code, info in STOCKS.items():
    print(f'\n处理股票: {info["name"]} ({ts_code})')
    print('-' * 60)
    
    try:
        # 加载数据
        df_stock = load_stock_data(
            ts_code,
            start_date=STRATEGY_PARAMS['start_date'],
            end_date=STRATEGY_PARAMS['end_date']
        )
        
        # 计算技术指标
        if STRATEGY_PARAMS['ma_type'] == 'MA':
            df_stock['ma_short'] = calculate_ma(df_stock['close'], STRATEGY_PARAMS['short_window'])
            df_stock['ma_long'] = calculate_ma(df_stock['close'], STRATEGY_PARAMS['long_window'])
        else:
            df_stock['ma_short'] = calculate_ema(df_stock['close'], STRATEGY_PARAMS['short_window'])
            df_stock['ma_long'] = calculate_ema(df_stock['close'], STRATEGY_PARAMS['long_window'])
        
        df_stock['atr'] = calculate_atr(df_stock, STRATEGY_PARAMS['atr_window'])
        
        if STRATEGY_PARAMS['trend_filter']:
            df_stock['ma_trend'] = calculate_ma(df_stock['close'], STRATEGY_PARAMS['trend_window'])
        
        # 生成信号
        df_stock = generate_signals(df_stock, STRATEGY_PARAMS)
        
        # 执行回测
        result = run_backtest(df_stock, STRATEGY_PARAMS)
        bh = run_buy_hold_backtest(df_stock, STRATEGY_PARAMS)
        
        # 计算指标
        strategy_return = (result['final_value'] - STRATEGY_PARAMS['initial_capital']) / STRATEGY_PARAMS['initial_capital']
        bh_return = (bh['final_value'] - STRATEGY_PARAMS['initial_capital']) / STRATEGY_PARAMS['initial_capital']
        excess = strategy_return - bh_return
        
        strategy_sharpe = calculate_sharpe_ratio(result['returns'])
        bh_sharpe = calculate_sharpe_ratio(bh['returns'])
        
        strategy_max_dd = calculate_max_drawdown(result['nav'])
        
        # 保存结果
        results.append({
            '股票': info['name'],
            '行业': info['industry'],
            '策略收益率(%)': round(strategy_return * 100, 2),
            'BUY-HOLD收益率(%)': round(bh_return * 100, 2),
            '超额收益(%)': round(excess * 100, 2),
            '策略夏普比率': round(strategy_sharpe, 2),
            'BUY-HOLD夏普比率': round(bh_sharpe, 2),
            '策略最大回撤(%)': round(strategy_max_dd * 100, 2),
            '交易次数': len(result['trades'])
        })
        
        print(f'✅ {info["name"]} 回测完成')
        print(f'  策略收益率: {strategy_return*100:.2f}%')
        print(f'  BUY-HOLD收益率: {bh_return*100:.2f}%')
        
    except Exception as e:
        print(f'❌ {info["name"]} 回测失败: {str(e)}')

# 生成汇总报告
print('\n' + '=' * 60)
print('📊 汇总对比表')
print('=' * 60)

summary_df = pd.DataFrame(results)
display(summary_df)

# 保存汇总报告
output_file = 'outputs/ma_backtest/reports/summary_report.csv'
os.makedirs(os.path.dirname(output_file), exist_ok=True)
summary_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f'\n✅ 汇总报告已保存: {output_file}')

print('=' * 60)

## 🎯 Cell 10: 参数优化（可选）

**网格搜索寻找最优参数组合**

In [ ]:
# ============================================================================
# 参数优化（网格搜索）
# ============================================================================

print('=' * 60)
print('🎯 参数优化（网格搜索）')
print('=' * 60)
print('\n⚠️ 注意：参数优化可能需要较长时间，请耐心等待...')

# 定义参数网格
param_grid = {
    'short_window': [5, 10, 15],
    'long_window': [20, 30, 50],
}

print(f'\n参数网格:')
print(f'  短均线周期: {param_grid["short_window"]}')
print(f'  长均线周期: {param_grid["long_window"]}')
print(f'  总组合数: {len(param_grid["short_window"]) * len(param_grid["long_window"])}')

# 网格搜索
optimization_results = []

for short in param_grid['short_window']:
    for long in param_grid['long_window']:
        if short >= long:
            continue
        
        # 更新参数
        test_params = STRATEGY_PARAMS.copy()
        test_params['short_window'] = short
        test_params['long_window'] = long
        
        # 重新计算技术指标
        df_test = df.copy()
        df_test['ma_short'] = calculate_ma(df_test['close'], short)
        df_test['ma_long'] = calculate_ma(df_test['close'], long)
        df_test['atr'] = calculate_atr(df_test, test_params['atr_window'])
        
        if test_params['trend_filter']:
            df_test['ma_trend'] = calculate_ma(df_test['close'], test_params['trend_window'])
        
        # 生成信号
        df_test = generate_signals(df_test, test_params)
        
        # 执行回测
        result = run_backtest(df_test, test_params)
        
        # 计算收益率
        total_return = (result['final_value'] - test_params['initial_capital']) / test_params['initial_capital']
        sharpe = calculate_sharpe_ratio(result['returns'])
        max_dd = calculate_max_drawdown(result['nav'])
        
        optimization_results.append({
            'short_window': short,
            'long_window': long,
            'total_return': total_return,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_dd,
            'num_trades': len(result['trades'])
        })

# 排序找到最优参数
results_df = pd.DataFrame(optimization_results)
results_df = results_df.sort_values('sharpe_ratio', ascending=False)

print('\n' + '=' * 60)
print('🏆 参数优化结果（按夏普比率排序）')
print('=' * 60)
display(results_df.head(10))

# 推荐最优参数
best_params = results_df.iloc[0]
print('\n💡 推荐参数组合:')
print(f'  短均线周期: {int(best_params["short_window"])}')
print(f'  长均线周期: {int(best_params["long_window"])}')
print(f'  夏普比率: {best_params["sharpe_ratio"]:.2f}')
print(f'  总收益率: {best_params["total_return"]*100:.2f}%')
print(f'  最大回撤: {best_params["max_drawdown"]*100:.2f}%')

print('=' * 60)